# CWS residual-risk model — leakage-audited runner

This notebook reruns the residual-risk stage with stricter feature modes. The earlier `operational_no_direct_residual` model is useful only as a diagnostic because it allows `temp_raw`, `ref_mu`, and `ref_sigma`; since the label is based on `abs((temp_raw - ref_mu) / ref_sigma)`, it can reconstruct the target.

Paper-facing models here are:

- `context_history`: metadata + time + meteorology + satellite + reference-support geometry + **train-only station residual history**.
- `context_static_met`: same, but without station residual history.

The notebook saves a feature leakage audit CSV so each run can be checked before using it in paper tables.


In [ ]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 220)
pd.set_option("display.max_rows", 160)
pd.set_option("display.width", 260)


## 1. Project setup


In [ ]:
cwd = os.getcwd()
cwd_main = os.path.abspath(os.path.join(cwd, os.pardir))
os.chdir(cwd_main)
print("cwd_main:", cwd_main)

import config_tool as cfm

city = os.environ.get("QC_PROJECT_ID", "project_id")
cwd_project = Path(cfm.cwd_data) / city
os.chdir(cwd_project)
print("cwd_project:", cwd_project)

sys.modules.pop("config_project", None)
import config_project as cfp

cwd_data_qc_results = Path(cfp.cwd_results_qc)
benchmark_dir = cwd_data_qc_results / "qc_benchmark"
reference_dir = benchmark_dir / "ows_reference"
residual_risk_dir = benchmark_dir / "residual_risk_audited"
reference_dir.mkdir(parents=True, exist_ok=True)
residual_risk_dir.mkdir(parents=True, exist_ok=True)

print("reference_dir:", reference_dir)
print("residual_risk_dir:", residual_risk_dir)


## 2. Import the audited residual-risk script


In [ ]:


script_dir = Path(cfm.cwd_scripts_preprocesing)
if str(script_dir) not in sys.path:
    sys.path.insert(0, str(script_dir))

import importlib
import qc_cws_residual_risk_model_audited as rrm
rrm = importlib.reload(rrm)
print("rrm:", rrm.__file__)


## 3. Select the upstream OWS-reference run


In [ ]:
reference_run_label = "catboost_corrected_ows_metadata_iteration01"
reference_method = "catboost"
calibration_mode = "time_train"

reference_run_dir = reference_dir / reference_run_label
reference_manifest_file = reference_run_dir / f"{cfp.city}_ows_reference_manifest_{reference_method}_{calibration_mode}.json"

reference_manifest = None
if reference_manifest_file.exists():
    with open(reference_manifest_file, "r") as f:
        reference_manifest = json.load(f)
    cws_paths = reference_manifest.get("cws_reference_paths_by_calibration_mode", {})
    CWS_REFERENCE_PATH = Path(cws_paths.get(calibration_mode) or reference_manifest.get("cws_primary_reference_path"))
    print("Loaded upstream manifest:", reference_manifest_file)
else:
    CWS_REFERENCE_PATH = reference_run_dir / f"{cfp.city}_cws_reference_residual_features_{reference_method}_{calibration_mode}.parquet"
    print("Upstream manifest not found. Using fallback path:", CWS_REFERENCE_PATH)

print("CWS_REFERENCE_PATH:", CWS_REFERENCE_PATH)
print("exists:", CWS_REFERENCE_PATH.exists())
if not CWS_REFERENCE_PATH.exists():
    raise FileNotFoundError("Update reference_run_label/reference_method/calibration_mode or set CWS_REFERENCE_PATH manually.")


## 4. Quick input sanity check


In [ ]:
df_raw = rrm.load_dataframe_auto(CWS_REFERENCE_PATH)
print("rows:", len(df_raw))
print("columns:", len(df_raw.columns))
print("stations:", df_raw[["network", "station_id"]].drop_duplicates().shape[0])
print("date range:", pd.to_datetime(df_raw["date"], utc=True).min(), "to", pd.to_datetime(df_raw["date"], utc=True).max())

required = ["date", "station_id", "network", "temp_raw", "ref_mu", "ref_sigma", "cws_ref_resid", "abs_cws_ref_resid", "cws_ref_z", "target_ref_risk"]
missing = [c for c in required if c not in df_raw.columns]
print("missing required columns:", missing)
if missing:
    raise KeyError(f"Missing required columns: {missing}")


calc_z = (pd.to_numeric(df_raw["temp_raw"], errors="coerce") - pd.to_numeric(df_raw["ref_mu"], errors="coerce")) / pd.to_numeric(df_raw["ref_sigma"], errors="coerce")
max_z_diff = np.nanmax(np.abs(calc_z - pd.to_numeric(df_raw["cws_ref_z"], errors="coerce")))
print("max |computed_z - cws_ref_z|:", max_z_diff)


## 5. Optional audit of an existing residual-risk result folder

Point this to the previous `residual_risk` output folder if you want to document which features were problematic.


In [ ]:
CURRENT_RESIDUAL_RISK_RESULTS_DIR = residual_risk_dir.parent / "residual_risk" / f"{reference_run_label}__residual_risk__{reference_method}__{calibration_mode}"
feature_audit_path = CURRENT_RESIDUAL_RISK_RESULTS_DIR / f"{city}_feature_audit.json"

if feature_audit_path.exists():
    with open(feature_audit_path, "r") as f:
        old_audit = json.load(f)
    rows = []
    for mode, a in old_audit.items():
        audit_df = rrm.feature_leakage_audit_table(a.get("feature_cols", []), mode)
        rows.append(audit_df)
        print("MODE:", mode)
        print(audit_df["severity"].value_counts(dropna=False))
        display(audit_df[audit_df["severity"].isin(["fatal", "diagnostic"])].head(60))
    old_leakage_audit = pd.concat(rows, ignore_index=True)
else:
    print("No previous feature audit found at", feature_audit_path)
    old_leakage_audit = None


## 6. Configure the paper-facing audited run


In [ ]:


RUN_REFERENCE_AVAILABLE_DIAGNOSTIC = False

feature_modes = ["context_history", "context_static_met"]
if RUN_REFERENCE_AVAILABLE_DIAGNOSTIC:
    feature_modes.append("reference_available_diagnostic")

risk_config = rrm.ResidualRiskConfig(
    city=city,
    input_path=str(CWS_REFERENCE_PATH),
    output_dir=str(residual_risk_dir),
    run_label=f"{reference_run_label}__residual_risk_AUDITED__{reference_method}__{calibration_mode}",
    overwrite_existing_run=False,

    reference_method=reference_method,
    calibration_mode=calibration_mode,
    source_reference_run_label=reference_run_label,
    source_reference_manifest_path=str(reference_manifest_file) if reference_manifest_file.exists() else None,

    split_strategy="time",
    valid_start="2021-10-01",
    test_start="2021-11-01",
    target_mode="auto",

    feature_modes=",".join(feature_modes),
    include_coordinates=False,
    include_qc_flags=False,
    drop_numeric_landcover_codes=True,

    run_catboost=True,
    iterations=1200,
    learning_rate=0.04,
    depth=8,
    l2_leaf_reg=8.0,
    auto_class_weights="Balanced",
    early_stopping_rounds=100,
    thread_count=None,
    used_ram_limit="42gb",
    use_gpu=False,
    verbose=100,
    max_train_rows=None,
    max_valid_rows=None,
)
print(risk_config)


## 7. Run audited residual-risk pipeline


In [ ]:
results = rrm.run_residual_risk_pipeline(cfg=risk_config)

print("Output dir:", results["output_dir"])
print("Scored output:", results.get("scored_path"))
print("Metrics:", results.get("metrics_path"))
print("Retention curves:", results.get("retention_path"))
print("Feature audit:", results.get("feature_audit_path"))
print("Leakage audit:", results.get("feature_leakage_audit_path"))
print("Manifest:", results["manifest_path"])


## 8. Review metrics and leakage audit


In [ ]:
split_counts = pd.read_csv(results["split_counts_path"])
metrics = pd.read_csv(results["metrics_path"])
reliability = pd.read_csv(results["reliability_path"])
retention = pd.read_csv(results["retention_path"])
band_summary = pd.read_csv(results["band_summary_path"])
leakage_audit = pd.read_csv(results["feature_leakage_audit_path"])

print("Split counts")
display(split_counts)
print("Metrics")
display(metrics.sort_values(["model", "split"]))
print("Band summary")
display(band_summary.sort_values(["method", "band"]))
print("Leakage audit counts")
display(leakage_audit.groupby(["feature_mode", "severity"]).size().reset_index(name="n"))
print("Fatal/diagnostic features — should be empty for paper-facing modes")
display(leakage_audit[leakage_audit["severity"].isin(["fatal", "diagnostic"])])


## 9. Retention-vs-error plots


In [ ]:
if len(retention):
    plt.figure(figsize=(8, 5))
    for method, g in retention.groupby("method"):
        gg = g.sort_values("retention_actual")
        plt.plot(gg["retention_actual"], gg["residual_mae"], marker="o", label=method)
    plt.xlabel("Retention fraction")
    plt.ylabel("Residual MAE against OWS reference (°C)")
    plt.title("Audited residual-risk: retention vs MAE")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

    plt.figure(figsize=(8, 5))
    for method, g in retention.groupby("method"):
        gg = g.sort_values("retention_actual")
        plt.plot(gg["retention_actual"], gg["residual_p95_abs"], marker="o", label=method)
    plt.xlabel("Retention fraction")
    plt.ylabel("p95 absolute residual against OWS reference (°C)")
    plt.title("Audited residual-risk: retention vs p95 absolute residual")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()


## 10. Reliability plots


In [ ]:
if len(reliability):
    for model_name, g_model in reliability.groupby("model"):
        plt.figure(figsize=(6, 5))
        for split, g in g_model.groupby("split"):
            gg = g[g["n"] > 0].sort_values("mean_pred_prob")
            plt.plot(gg["mean_pred_prob"], gg["event_rate"], marker="o", label=split)
        plt.plot([0, 1], [0, 1], linestyle="--")
        plt.xlabel("Mean predicted probability")
        plt.ylabel("Observed high-confidence residual-risk rate")
        plt.title(f"Reliability: {model_name}")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()


## 11. Save a compact paper summary


In [ ]:
summary_rows = []
for model, g in metrics.groupby("model"):
    test = g[g["split"] == "test"]
    if len(test):
        row = test.iloc[0].to_dict()
        summary_rows.append(row)
summary = pd.DataFrame(summary_rows)
summary_path = Path(results["output_dir"]) / f"{city}_audited_residual_risk_paper_summary.csv"
summary.to_csv(summary_path, index=False)
print("summary_path:", summary_path)
display(summary)
